# GCascadeV5 Tutorial Notebook

Welcome to the GCascadeV5 tutorial. This notebook is intended to be beginner-friendly and mirrors the spirit of the V4 tutorial: we explain *what each function does*, *what units it expects*, and *how to format arrays correctly*.

This notebook focuses on core propagation functionality first (point, diffuse, evolving), then advanced controls (EBL model and magnetic field), and finally parity workflow against V4.

## 0) Prerequisites and execution notes

- Run notebook cells in order.
- Inputs and outputs use **physical units** (detailed below).
- All source redshifts must satisfy `0 <= z <= 10`.
- Before running attenuation/cascade functions, confirm that GCascadeV5 points to your local `LibrariesV4` directory.


## 1) Configure library paths (first step)

GCascadeV5 uses two configurable paths:

- `library_path`: read-only V4 precomputed tables (`LibrariesV4`).
- `generated_library_path`: where V5 writes generated cycle tables (used by `changeMagneticField`).

You can configure paths in either way:

1. **Environment variables before import**
   - `GCASCADE_LIB_PATH=/absolute/path/to/LibrariesV4`
   - `GCASCADE_GENERATED_LIB_PATH=/absolute/path/to/output_folder`
2. **Runtime setters after import**
   - `gc.set_library_path(...)`
   - `gc.set_generated_library_path(...)`

The next cell prints active paths so you can verify configuration immediately.


In [ ]:
import os
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# Option A: set environment variables BEFORE importing gcascade_v5
# os.environ['GCASCADE_LIB_PATH'] = '/absolute/path/to/LibrariesV4'
# os.environ['GCASCADE_GENERATED_LIB_PATH'] = '/absolute/path/to/generated_libraries'

import gcascade_v5 as gc

# Option B: configure paths AFTER import (uncomment and edit as needed)
# gc.set_library_path('/absolute/path/to/LibrariesV4')
# gc.set_generated_library_path('/absolute/path/to/generated_libraries')

print('Active V4 library path:', gc.get_library_path())
print('Active generated-library path:', gc.get_generated_library_path())

if not Path(gc.get_library_path()).exists():
    print('WARNING: library path does not exist yet. Set it with gc.set_library_path(...)')


## 2) Core arrays and what they mean

GCascadeV5 exposes key grids used by all computations:

- `energies`: gamma-ray energy grid in **GeV**.
  - Length: 300
  - Log-spaced from `1e-1 GeV` to `1e12 GeV`
- `diffuseDistances`: redshift grid used for source-population integrals and propagation slicing.
  - Length: 1036
- `zReg`: coarser redshift regions (`0` to `10` in steps of `0.01`) used to index precomputed interaction tables.

In [ ]:
print('len(energies) =', len(gc.energies))
print('energies[0], energies[-1] =', gc.energies[0], gc.energies[-1], 'GeV')
print('len(diffuseDistances) =', len(gc.diffuseDistances))
print('diffuseDistances min/max =', gc.diffuseDistances[0], gc.diffuseDistances[-1])
print('len(zReg) =', len(gc.zReg))
print('current EBL index =', gc.EBLindex)

## 3) Physical unit conventions

### Injected spectra
For point and diffuse non-evolving functions, the injected spectrum should be:

- `dN/dE(E)` in **GeV^-1 s^-1**
- Evaluated on `gc.energies`
- Shape `(300,)`

### Source-density distribution
For diffuse/evolving calculations, `zDistrib` should be:

- `rho(z)` in **cm^-3**
- Evaluated on `gc.diffuseDistances`
- Shape `(1036,)`

### Typical output units
- Point-source outputs: **GeV^-1 s^-1 cm^-2**
- Diffuse / evolving outputs: **GeV^-1 s^-1 cm^-2 sr^-1**

## 4) Build a valid injected spectrum

`cutoffPowerLaw(E, gamma, cutoff, amp)` is a convenience helper:

`dN/dE = amp * E^(-gamma) * exp(-E/cutoff)`

where `E` and `cutoff` are in GeV.

In [ ]:
inj = gc.cutoffPowerLaw(gc.energies, gamma=2.2, cutoff=1e7, amp=1e40)
print('shape:', inj.shape)
print('min/max:', inj.min(), inj.max())

In [ ]:
fig, ax = gc.specPlot(inj)
ax.set_title('Injected spectrum: E^2 dN/dE')

## 5) Point-source propagation

For a source at redshift `z_source`, GCascade provides three levels:

- `RedshiftPoint(inj, z_source)`: only cosmological redshifting
- `AttenuatePoint(inj, z_source)`: redshifting + pair-production attenuation
- `CascadePoint(inj, z_source)`: redshifting + attenuation + electromagnetic cascade regeneration

In [ ]:
z_source = 0.3

phi_redshift = gc.RedshiftPoint(inj, z_source)
phi_atten = gc.AttenuatePoint(inj, z_source)
phi_cascade = gc.CascadePoint(inj, z_source)

print(phi_redshift.shape, phi_atten.shape, phi_cascade.shape)

In [ ]:
plt.figure(figsize=(7, 5))
plt.loglog(gc.energies, gc.energies**2 * phi_redshift, label='RedshiftPoint')
plt.loglog(gc.energies, gc.energies**2 * phi_atten, label='AttenuatePoint')
plt.loglog(gc.energies, gc.energies**2 * phi_cascade, label='CascadePoint')
plt.xlabel('E [GeV]')
plt.ylabel('E^2 Phi(E)')
plt.legend()
plt.grid(alpha=0.25, which='both')
plt.title('Point-source transport comparison')

## 6) Diffuse non-evolving source population

Diffuse functions assume all sources share one intrinsic spectrum (`inj`) and are distributed in redshift via `zDistrib`.

Required shapes:
- `inj`: `(300,)`
- `zDistrib`: `(1036,)`

Main APIs:
- `RedshiftDiffuse(inj, z_max, zDistrib)`
- `AttenuateDiffuse(inj, z_max, zDistrib)`
- `CascadeDiffuse(inj, z_max, zDistrib)`

In [ ]:
z = gc.diffuseDistances
z_distrib = 1e-9 * np.exp(-((z - 1.0) / 0.7) ** 2)  # cm^-3

phi_diff_r = gc.RedshiftDiffuse(inj, 3.0, z_distrib)
phi_diff_a = gc.AttenuateDiffuse(inj, 3.0, z_distrib)
phi_diff_c = gc.CascadeDiffuse(inj, 3.0, z_distrib)

print(phi_diff_r.shape, phi_diff_a.shape, phi_diff_c.shape)

## 7) Diffuse evolving source population

For evolving populations, the injection is a 2D array:

- `inj2d[z_index, e_index] = dN/dE(E, z)`
- Shape `(len(diffuseDistances), len(energies)) = (1036, 300)`

Main APIs:
- `RedshiftEvolving(inj2d, z_max, zDistrib)`
- `AttenuateEvolving(inj2d, z_max, zDistrib)`
- `CascadeEvolving(inj2d, z_max, zDistrib)`

In [ ]:
inj2d = np.empty((len(gc.diffuseDistances), len(gc.energies)))
for i, zi in enumerate(gc.diffuseDistances):
    gamma_i = 2.0 + 0.2 * zi / 10.0
    inj2d[i] = gc.cutoffPowerLaw(gc.energies, gamma=gamma_i, cutoff=1e7, amp=1e40)

phi_ev_r = gc.RedshiftEvolving(inj2d, 3.0, z_distrib)
phi_ev_a = gc.AttenuateEvolving(inj2d, 3.0, z_distrib)
phi_ev_c = gc.CascadeEvolving(inj2d, 3.0, z_distrib)

print(phi_ev_r.shape, phi_ev_a.shape, phi_ev_c.shape)

## 8) Advanced controls

### 8.1 Change EBL model
EBL index map:
- `0`: CMB only
- `1`: Saldana-Lopez 2021 (default)
- `2`: Saldana-Lopez high
- `3`: Saldana-Lopez low
- `4`: Finke 2022
- `5`: Franceschini & Rodighiero 2018
- `6`: Dominguez 2011

In [ ]:
old_ebl = gc.EBLindex
if old_ebl != 4:
    gc.changeEBLModel(4)
print('active EBL:', gc.EBLindex)

# restore
if gc.EBLindex != old_ebl:
    gc.changeEBLModel(old_ebl)
print('restored EBL:', gc.EBLindex)

### 8.2 Change magnetic field
`changeMagneticField(Bfield, gamma, EBLindex)` regenerates cycle tables for the selected EBL model.

- `Bfield` in **Gauss**
- field scaling: `B(z) = Bfield * (1+z)^gamma`
- this is computationally heavier than regular flux calls
- generated tables are written under `generated_libraries/`

In [ ]:
# Uncomment to run when needed (can take time).
# gc.changeMagneticField(1e-16, 2.0, 1)

## 9) Exporting results

A common workflow is to export `(energy, flux)` columns for external plotting/fitting.

In [ ]:
point_table = np.column_stack([gc.energies, phi_cascade])
np.savetxt('point_cascade_flux.csv', point_table, delimiter=',', header='E_GeV,Phi_GeV^-1_s^-1_cm^-2', comments='')
print('saved point_cascade_flux.csv')

## 10) Numerical parity workflow against V4

Use this after exporting fixture cases from Mathematica:

1. Export fixtures with `scripts/export_v4_reference_template.wl`.
2. Place them in `benchmarks/v4_reference/<case_name>/` with `meta.json`, inputs, and `expected.csv`.
3. Run parity check from terminal:

```bash
cd /Users/antonio/Desktop/Research/GCascadeV5
python3 scripts/run_parity.py
```

Default acceptance is relative tolerance `1e-3` with a small absolute floor near zero.